# Music Generation with AI — CodeAlpha Task 3

An LSTM network trained on J.S. Bach's four-part chorales, taught to compose new melodic lines in a similar style.

**Pipeline:**
1. Collect MIDI-equivalent score data (Bach chorales, bundled with `music21`)
2. Preprocess into note/duration sequences
3. Build an LSTM model
4. Train it on the sequences
5. Generate a new sequence and export it as a playable MIDI file

Run the cells top to bottom. On Google Colab: **Runtime → Change runtime type → GPU** first, so training is fast.

In [ ]:
# music21 isn't preinstalled on Colab; TensorFlow and numpy usually are.
!pip install -q --break-system-packages music21 midi2audio 2>/dev/null || pip install -q music21 midi2audio
!apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null

In [ ]:
import warnings, random, os
from fractions import Fraction
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from music21 import corpus, stream, note, tempo
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

## 1. Collect MIDI music data

Rather than pointing at an external archive (links rot, datasets get taken down), this uses the **433 Bach chorales bundled directly inside `music21`** — real classical repertoire, public domain, zero download required.

Each chorale is written for four voices (soprano, alto, tenor, bass). We pull out just the **soprano line** for a clean, single melody — easier to model well than full four-part harmony with a small dataset, and still very recognisably Bach.

In [ ]:
def get_soprano(score):
    # Return the soprano part by name; falls back to the first part if not found
    # (a few chorales list a doubling instrument like 'Horn' before the voices).
    for part in score.parts:
        if 'soprano' in str(part.id).lower():
            return part
    return score.parts[0]

def score_to_events(part):
    # Convert a music21 part into a list of 'pitch:duration' string tokens.
    events = []
    for el in part.flatten().notesAndRests:
        dur = str(el.duration.quarterLength)
        if el.isRest:
            events.append(f"REST:{dur}")
        elif el.isChord:
            top = max(el.pitches, key=lambda p: p.midi)
            events.append(f"{top.midi}:{dur}")
        else:
            events.append(f"{el.pitch.midi}:{dur}")
    return events

paths = corpus.getComposer('bach')
print(f"Found {len(paths)} Bach pieces in the bundled corpus")

all_events = []
for p in paths:
    try:
        score = corpus.parse(p)
        events = score_to_events(get_soprano(score))
        if len(events) > 20:
            all_events.append(events)
    except Exception:
        continue

print(f"Successfully parsed {len(all_events)} chorales")
print(f"Total notes/rests collected: {sum(len(e) for e in all_events)}")

## 2. Preprocess into note sequences

Each event is encoded as a token combining **pitch and duration**, e.g. `"67:1.0"` means a G4 held for one quarter note. Rests become `"REST:<duration>"`. Keeping pitch and rhythm in one token means a single model can learn both at once.

The model is trained to predict the *next* token from the previous `SEQ_LEN` tokens. Sliding windows are built separately within each chorale so training examples never blend two different pieces together.

In [ ]:
SEQ_LEN = 40

vocab = sorted(set(e for piece in all_events for e in piece))
token_to_idx = {t: i for i, t in enumerate(vocab)}
idx_to_token = {i: t for t, i in token_to_idx.items()}
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size} unique pitch:duration tokens")

X, y = [], []
for piece in all_events:
    encoded = [token_to_idx[t] for t in piece]
    for i in range(len(encoded) - SEQ_LEN):
        X.append(encoded[i:i + SEQ_LEN])
        y.append(encoded[i + SEQ_LEN])

X = np.array(X)
y = np.array(y)
print(f"Training sequences: {X.shape[0]}, each {SEQ_LEN} steps long")

## 3. Build the LSTM model

`Embedding` turns each token index into a learned vector. Two stacked `LSTM` layers pick up short- and longer-range melodic patterns, and the final `Dense` layer outputs a probability distribution over every possible next token.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(SEQ_LEN,)),
    layers.Embedding(vocab_size, 100),
    layers.LSTM(256, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(256),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(vocab_size, activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 4. Train the model

`EarlyStopping` halts training once the loss stops improving and restores the best weights; `ModelCheckpoint` saves that best version to disk. On a Colab GPU this typically finishes in a few minutes; on CPU, budget 15–20.

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
checkpoint = callbacks.ModelCheckpoint('best_model.keras', monitor='loss', save_best_only=True)

history = model.fit(
    X, y,
    batch_size=128,
    epochs=100,
    validation_split=0.1,
    callbacks=[early_stop, checkpoint],
    verbose=2,
)

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('Training progress')
plt.show()

## 5. Generate a new sequence and save it as MIDI

Starting from a random seed lifted from the training data, the model predicts one token at a time and feeds each prediction back in as input for the next step. Sampling with **temperature** — instead of always taking the single most likely token — keeps the output from looping on one note.

In [ ]:
def sample_with_temperature(probabilities, temperature=0.8):
    probabilities = np.asarray(probabilities).astype('float64')
    probabilities = np.log(probabilities + 1e-9) / temperature
    probabilities = np.exp(probabilities) / np.sum(np.exp(probabilities))
    return np.random.choice(len(probabilities), p=probabilities)

def generate_sequence(model, seed_tokens, num_steps=200, temperature=0.8):
    generated = list(seed_tokens)
    window = list(seed_tokens)
    for _ in range(num_steps):
        x = np.array(window[-SEQ_LEN:]).reshape(1, SEQ_LEN)
        probs = model.predict(x, verbose=0)[0]
        next_idx = sample_with_temperature(probs, temperature)
        generated.append(next_idx)
        window.append(next_idx)
    return generated

seed_piece = random.choice([p for p in all_events if len(p) > SEQ_LEN])
start = random.randint(0, len(seed_piece) - SEQ_LEN - 1)
seed = [token_to_idx[t] for t in seed_piece[start:start + SEQ_LEN]]

generated_indices = generate_sequence(model, seed, num_steps=200, temperature=0.8)
generated_tokens = [idx_to_token[i] for i in generated_indices]
print("First 20 generated tokens:", generated_tokens[:20])

In [ ]:
def tokens_to_midi(tokens, out_path, bpm=90):
    s = stream.Stream()
    s.append(tempo.MetronomeMark(number=bpm))
    for tok in tokens:
        pitch_part, dur_part = tok.rsplit(':', 1)
        qlen = float(Fraction(dur_part))
        if pitch_part == 'REST':
            el = note.Rest()
        else:
            el = note.Note()
            el.pitch.midi = int(pitch_part)
        el.duration.quarterLength = qlen
        s.append(el)
    s.write('midi', fp=out_path)
    return out_path

midi_path = tokens_to_midi(generated_tokens, 'generated_bach_style.mid')
print(f"Saved: {midi_path}")

## Optional: listen without leaving the notebook

This renders the MIDI to a WAV using a General MIDI soundfont and plays it inline. If it errors out (synthesizer setup varies a little by environment), just download `generated_bach_style.mid` from the file browser on the left and open it in any media player, DAW, or online MIDI player instead.

In [ ]:
import glob
from midi2audio import FluidSynth
from IPython.display import Audio, display

soundfonts = glob.glob('/usr/share/sounds/sf2/FluidR3_GM.sf2') or glob.glob('/usr/share/sounds/sf2/*.sf2')
if soundfonts:
    fs = FluidSynth(sound_font=soundfonts[0])
    fs.midi_to_audio('generated_bach_style.mid', 'generated_bach_style.wav')
    display(Audio('generated_bach_style.wav'))
else:
    print("No soundfont found on this machine — download the .mid file and play it externally instead.")

## Recap

- **Data**: 433 Bach chorale soprano lines, bundled with `music21` — no external download needed
- **Preprocessing**: each note/rest becomes a `pitch:duration` token, roughly 210 unique tokens total
- **Model**: Embedding → 2× LSTM(256) → Dense softmax, trained with early stopping
- **Generation**: temperature-sampled, 200 new tokens grown from a random seed out of the training set
- **Output**: `generated_bach_style.mid` — playable in any MIDI player, DAW, or the optional cell above

Ways to push it further: train on the full four-part texture instead of just the soprano line, add an attention layer on top of the LSTMs, or swap the LSTM for a GAN, as the task brief suggests as an alternative architecture.